In [1]:
import torch

In [2]:
doc_ids = torch.tensor([[0,0,0,1,1,2,2,2,2]])

In [5]:
batch_size, seq_len = doc_ids.shape

In [6]:
is_new_group = torch.cat(
    [
        torch.ones_like(doc_ids[:, :1], dtype=torch.bool),
        doc_ids[:, 1:] != doc_ids[:, :-1],
    ],
    dim=1,
)

In [7]:
# Get the indices where new groups start
group_start_indices = torch.where(is_new_group)[1]
group_start_indices = group_start_indices.view(batch_size, -1)

# Broadcast subtraction: subtract the start index of current group from position
positions = (
    torch.arange(doc_ids.size(1), device=doc_ids.device)
    .unsqueeze(0)
    .expand_as(doc_ids)
)

In [8]:
group_starts = torch.zeros_like(positions)
group_starts[:, is_new_group[0]] = group_start_indices
group_starts = group_starts.cummax(dim=1)[0]  # Forward fill the start indices
positions = positions - group_starts

In [9]:
positions

tensor([[0, 1, 2, 0, 1, 0, 1, 2, 3]])

In [11]:
positions.max() <= 128

tensor(True)